# 0. import

In [ ]:
# imported variables: PRETRAIN_CKPT_DIR, TRACKS_GM12878_CHR10_NPY, REPO,
#     OUT_DIR (configure them in paths.py)
# --- plot_revision setup ---
import sys
from pathlib import Path
_HERE = Path.cwd().resolve()
for cand in (_HERE, *_HERE.parents):           # repo root = first folder with paths.py
    if (cand / "paths.py").exists():
        sys.path.insert(0, str(cand))
        break
from paths import (
    REPO, ensure_out_dir,
 CLADE_DIR, TREE_NWK,

    CKPT_ROOT, EPI_EVO2HIC_DIR, PRETRAIN_CKPT_DIR, PRETRAIN_CKPT,
    SR_EVO2HIC_HUMAN_DIR, SEQ2HIC_CKPT,
    HIC_DATA_ROOT, HIC_RAW_DIR, HIC_RAW, DNA_FASTA, MOTIF_MEME,
    EVO2_EMB_HUMAN, EVO2_EMB_MOUSE, TRACKS_GM12878_CHR10_NPY,
    ORCA_MALLPREDS_PTH, ORCA_EXPECTED_NPY,
    MOTIF_STATS_CSV,
    HUMAN_MERGED_CSV, HUMAN_INTER_DIR,
    HIC2TRACK_DIR, HIC2TRACK,
)
OUT_DIR = ensure_out_dir()
# Make `dataset.*` / `config` importable inside notebook cells
from paths import add_repo_to_syspath
add_repo_to_syspath()


In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.lines import Line2D

from importlib import reload
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import matplotlib.patches as patches

from plot_utils import colors

from scipy.stats import gaussian_kde, spearmanr
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-white')


In [ ]:
chromosomes = {
    'human': [9,10],
    'mouse': [1,2,3,4]
}

In [ ]:
fig = plt.figure(figsize=(8.27, 3))

gs = gridspec.GridSpec(
    3, 3,  
    figure=fig,
    height_ratios=[0.5, 2, 0.5],
    hspace=0.0,
    wspace=0.35
)

axes = []
for i, row in enumerate([1]):
    row_axes = []
    for j in range(3):
        ax = fig.add_subplot(gs[row, j])
        row_axes.append(ax)
    axes.append(row_axes)

axes = np.array(axes)
for ax in axes.flatten():
    ax.set_aspect('equal')
    # ax.axis('off')
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)


In [ ]:
import matplotlib as mpl
plt.rcParams.update({'font.size': 8})
print("xtick.labelsize:", mpl.rcParams['xtick.labelsize'])
print("ytick.labelsize:", mpl.rcParams['ytick.labelsize'])
print("axes.labelsize:", mpl.rcParams['axes.labelsize'])
print("axes.titlesize:", mpl.rcParams['axes.titlesize'])
print("legend.fontsize:", mpl.rcParams['legend.fontsize'])


# 3. Embedding umap

## 3.2 ploting

In [ ]:
import umap
def read(f, norm=True):
    emb = np.load(f)
    emb = emb.transpose(-1, -2)
    if norm:
        emb /= np.linalg.norm(emb, axis=-1, keepdims=True)
    emb = np.nan_to_num(emb)
    return emb

embeddings = read(str(PRETRAIN_CKPT_DIR / 'human_embeds/dna_emb_10.npy'))

# Step 1: Fit UMAP to reduce to 2D
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=100, min_dist=0.5, metric='cosine')
embeddings_umap = reducer.fit_transform(embeddings)


In [ ]:
a = np.load(str(TRACKS_GM12878_CHR10_NPY))
tracks = [
    'DNase', 
    'CTCF', 
    'H3K27ac',
    'H3K27me3',
    'H3K4me3'
]
print(a.shape)

In [ ]:
def kde_mean_2d(x, y, val, at=None, bw_method=None):
    """
    Local mean signal over 2-D positions (kernel regression).
    - x,y: point coordinates (1-D arrays)
    - val: signal of each point (may be negative)
    - at: evaluation positions (X, Y); if None, evaluate at the input points
          X, Y may be 1-D arrays (same length) or a grid (2-D arrays from np.meshgrid)
    - bw_method: bandwidth passed to gaussian_kde (e.g. 'scott'/'silverman' or a float factor)
    Returns: array of mean signal at the `at` positions
    """
    x = np.asarray(x); y = np.asarray(y); val = np.asarray(val)
    pts = np.vstack([x, y])

    kde_den = gaussian_kde(pts, bw_method=bw_method)                 # denominator: plain density
    kde_val = gaussian_kde(pts, weights=val, bw_method=bw_method)  # numerator (weighted)
    
    # evaluation positions
    if at is None:
        eval_xy = pts
        reshape_to = None
    else:
        X, Y = at
        if X.ndim == 2:  # grid
            eval_xy = np.vstack([X.ravel(), Y.ravel()])
            reshape_to = X.shape
        else:            # point set
            eval_xy = np.vstack([X, Y])
            reshape_to = None

    num = kde_val(eval_xy)
    den = kde_den(eval_xy)
    mean = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)

    if reshape_to:
        mean = mean.reshape(reshape_to)
    return mean


In [ ]:


from scipy.stats import rankdata
import matplotlib.gridspec as gridspec

L, R, D = 1000, -1000, 1

percentiles = rankdata(a, axis=-1) / a.shape[-1]

ftracks = ['Average'] + tracks

k = 0.1
boundaries = [0, k, 1-k, 1]

cmap = 'RdBu_r'
norm=None
# cmap = mcolors.ListedColormap(["blue", "lightgrey", "red"])
# norm = mcolors.BoundaryNorm(boundaries, cmap.N)

def plot_umap(embeddings_umap, c, name, ax):
    ax.clear()
    ax.set_aspect('auto')

    x = embeddings_umap[L:R:D, 0]
    y = embeddings_umap[L:R:D, 1]
    val = c[L:R:D]

    mean_at_pts = kde_mean_2d(x, y, val, bw_method=0.05)  # or 0.5/1.0 etc. to tune the bandwidth

    sc = ax.scatter(x, y, c=mean_at_pts, s=0.01, alpha=0.3, cmap=cmap, norm=norm, rasterized=True)

    sns.despine(ax=ax, bottom=True, left=True)
    # ax.set_xlabel('UMAP 0')
    # ax.set_ylabel('UMAP 1')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(name)

    # current axis limits
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    # length of the short axis arrows (fraction)
    frac = 0.3
    dx = (xmax - xmin) * frac
    dy = (ymax - ymin) * frac

    # short axis lines (bottom left)
    ax.plot([xmin, xmin + dx], [ymin, ymin], color='black', lw=0.8, clip_on=False)
    ax.plot([xmin, xmin], [ymin, ymin + dy], color='black', lw=0.8, clip_on=False)

    # arrow heads
    ax.annotate('', xy=(xmin + dx, ymin), xytext=(xmin + dx*0.6, ymin),
                arrowprops=dict(arrowstyle='-|>', lw=0.8, color='black', shrinkA=0, shrinkB=0))
    ax.annotate('', xy=(xmin, ymin + dy), xytext=(xmin, ymin + dy*0.6),
                arrowprops=dict(arrowstyle='-|>', lw=0.8, color='black', shrinkA=0, shrinkB=0))

    # optional: label next to the arrow
    ax.text(xmin, ymin-1, 'UMAP 0', va='center', fontsize=6)
    ax.text(xmin-1, ymin, 'UMAP 1', ha='center', rotation=90, fontsize=6)

    return sc

sc1 = plot_umap(embeddings_umap, percentiles.mean(axis=0), ftracks[0], axes[0][0])
sc2 = plot_umap(embeddings_umap, percentiles[1], tracks[1], axes[0][1])
sc3 = plot_umap(embeddings_umap, percentiles[4], tracks[4], axes[0][2])

fig

In [ ]:
# shared colorbar on the far right
cb = fig.colorbar(sc3, ax=axes[0].ravel().tolist(), location='right',
                  fraction=0.04, pad=0.02)  # fraction/pad can be tuned
cb.outline.set_linewidth(0.1)
cb.set_ticks([])
cb.set_label('Epigenomic signal intensity')

fig

In [ ]:
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i','j']
axs = [axes[0][0], axes[0][1], axes[0][2]]

for ax, label in zip(axs, labels):
    ax.text(
        -0.2, 1.2, f'{label}',           # position relative to the axes (top left)
        transform=ax.transAxes, 
        fontsize=12,
        fontname='Arial',               # font
        fontweight='bold',                  # bold (Nature style)
        ha='left', va='top'                 # alignment
    )
fig

In [ ]:
fig.savefig(str(OUT_DIR / 'supp_fig2.pdf'), dpi=600)